In [ ]:
# ============================================================
# 🐉 Chatbot RAG — Juego de Tronos (Libro 1)
# ============================================================
# Embedding  : Qwen/Qwen3-Embedding-8B  (SentenceTransformer)
# LLM        : google/gemma-3-27b-it
# VectorDB   : ChromaDB — 2 BDs separadas:
#                · chroma_db_resumenes_qwen  → colección "resumenes_got"
#                · libro_largo_qwen          → colección "libro"
# Alucinac.  : LettuceDetect (EuroBERT-610M ES)
# ============================================================
#
# CÓMO USAR EN JUPYTER / VS CODE:
#   Los bloques separados por  # %% [nombre]  son celdas.
#   Para ejecutar como script:  python got_chatbot_rag.py
# ============================================================


In [1]:
!pip uninstall -y transformers sentence-transformers lettucedetect tokenizers huggingface-hub

!pip install -q "transformers==4.46.3"
!pip install -q "tokenizers==0.20.3"
!pip install -q "huggingface-hub==0.26.5"
!pip install -q "sentence-transformers==3.3.1"
!pip install -q "lettucedetect==0.1.8"

Found existing installation: transformers 5.6.2
Uninstalling transformers-5.6.2:
  Successfully uninstalled transformers-5.6.2
Found existing installation: sentence-transformers 3.3.1
Uninstalling sentence-transformers-3.3.1:
  Successfully uninstalled sentence-transformers-3.3.1
Found existing installation: lettucedetect 0.1.8
Uninstalling lettucedetect-0.1.8:
  Successfully uninstalled lettucedetect-0.1.8
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: huggingface_hub 1.12.0
Uninstalling huggingface_hub-1.12.0:
  Successfully uninstalled huggingface_hub-1.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.26.5 which is incompatible.
diffusers 0.37.1 requires huggingface-h

In [ ]:
import os
os.kill(os.getpid(), 9)

: 

: 

In [2]:
import torch
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from lettucedetect.models.inference import HallucinationDetector

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
CHROMA_PATH_RESUMENES = "/content/drive/MyDrive/NLP_PRACTICA/Persist/chroma_db_resumenes_qwen"
CHROMA_PATH_LIBRO     = "/content/drive/MyDrive/NLP_PRACTICA/Persist/libro_largo_qwen"

# Nombres de colección tal como se crearon en el notebook de indexación
COLLECTION_RESUMENES = "resumenes_got"
COLLECTION_LIBRO     = "libro"

# Modelos
EMBED_MODEL  = "Qwen/Qwen3-Embedding-8B"
LLM_MODEL    = "google/gemma-4-31B-it"
HALLUC_MODEL = "KRLabsOrg/lettucedect-610m-eurobert-es-v1"

TOP_K_LIBRO     = 4   # chunks a recuperar del libro
TOP_K_RESUMENES = 2   # chunks a recuperar de los resumenes del fandom

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


Device: cuda


In [5]:

# %% [2. Cargar las dos ChromaDB]

# Dos clientes independientes, uno por BD
client_resumenes = chromadb.PersistentClient(path=CHROMA_PATH_RESUMENES)
client_libro     = chromadb.PersistentClient(path=CHROMA_PATH_LIBRO)

col_resumenes = client_resumenes.get_collection(COLLECTION_RESUMENES)
col_libro     = client_libro.get_collection(COLLECTION_LIBRO)

print(f"Coleccion resumenes fandom -> {col_resumenes.count()} chunks")
print(f"Coleccion libro            -> {col_libro.count()} chunks")


Coleccion resumenes fandom -> 320 chunks
Coleccion libro            -> 328 chunks


In [14]:
# %% [3. Modelo de embeddings (Qwen3-Embedding-8B — SentenceTransformer)]

# Usamos SentenceTransformer igual que en el notebook de indexacion,
# con normalize_embeddings=True para mantener consistencia con los vectores guardados.
embedder = SentenceTransformer(EMBED_MODEL)

def get_embedding(text: str) -> list:
    """Genera el embedding de un texto, identico a como se hizo al indexar."""
    return embedder.encode(text, normalize_embeddings=True).tolist()

print("Modelo de embeddings cargado")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 18.81 MiB is free. Including non-PyTorch memory, this process has 79.22 GiB memory in use. Of the allocated memory 78.61 GiB is allocated by PyTorch, and 128.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:

# %% [4. Modelo generador (Gemma-4)]

dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32

llm_pipe = pipeline(
    "text-generation",
    model=LLM_MODEL,
    device_map="auto",
    torch_dtype=dtype,
    trust_remote_code=True,
)

print("Modelo generador cargado")



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

Modelo generador cargado


In [7]:
# %% [5. Detector de alucinaciones (LettuceDetect)]

halluc_detector = HallucinationDetector(
    method="transformer",
    model_path=HALLUC_MODEL,
    lang="es",
    trust_remote_code=True,
)

print("Detector de alucinaciones cargado")


config.json: 0.00B [00:00, ?B/s]

configuration_eurobert.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/KRLabsOrg/lettucedect-610m-eurobert-es-v1:
- configuration_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/582 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.43G [00:00<?, ?B/s]

KeyError: 'default'

In [ ]:
# %% [6. Funciones del pipeline RAG]

SYSTEM_PROMPT = (
    "Eres un experto en el universo de Juego de Tronos, "
    "especializado en el primer libro de la saga 'Cancion de Hielo y Fuego'. "
    "Responde SOLO con la informacion que aparezca en los fragmentos del contexto. "
    "Si no encuentras la respuesta en el contexto, dilo explicitamente. "
    "Responde siempre en espanol."
)


def retrieve(query: str):
    """
    Recupera los chunks mas relevantes de ambas colecciones.

    NOTA sobre lo que devuelve 'documents' en cada coleccion:
    - col_resumenes: el documento es el parrafo del resumen (texto plano).
    - col_libro:     el documento es el retrieval_text enriquecido, que incluye
                     metadatos (POV, capitulo, personajes, ubicaciones...) + texto
                     del chunk. Es exactamente lo que se embebio al indexar, por
                     lo que el modelo lo recibe todo como contexto.
    """
    emb = get_embedding(query)

    res_libro = col_libro.query(
        query_embeddings=[emb],
        n_results=TOP_K_LIBRO,
        include=["documents", "metadatas"],
    )
    res_resumenes = col_resumenes.query(
        query_embeddings=[emb],
        n_results=TOP_K_RESUMENES,
        include=["documents", "metadatas"],
    )

    chunks_libro     = res_libro["documents"][0]      # lista de retrieval_texts
    chunks_resumenes = res_resumenes["documents"][0]  # lista de parrafos

    return chunks_libro, chunks_resumenes


def build_context(chunks_libro: list, chunks_resumenes: list) -> str:
    """Une los chunks de ambas fuentes en un bloque de contexto formateado."""
    partes = []
    if chunks_libro:
        partes.append(
            "### Fragmentos del libro\n" + "\n---\n".join(chunks_libro)
        )
    if chunks_resumenes:
        partes.append(
            "### Resumenes de capitulos (Fandom)\n" + "\n---\n".join(chunks_resumenes)
        )
    return "\n\n".join(partes)


def _call_llm(messages: list) -> str:
    """Llama al pipeline de Gemma y extrae el texto generado."""
    output    = llm_pipe(messages, max_new_tokens=512, do_sample=False,
                         temperature=None, top_p=None)
    generated = output[0]["generated_text"]
    # El pipeline chat devuelve la lista de mensajes completa;
    # el ultimo elemento es el turno del assistant.
    if isinstance(generated, list):
        return generated[-1]["content"]
    return generated


def generate_answer(question: str, context: str) -> str:
    """Genera la respuesta sin historial."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Contexto:\n{context}\n\nPregunta: {question}"},
    ]
    return _call_llm(messages)


def generate_answer_with_history(question: str, context: str, history: list) -> str:
    """
    Genera la respuesta incorporando el historial de conversacion.
    Ventana de 4 turnos para no saturar el contexto de Gemma.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history[-4:]:
        messages.append({"role": "user",      "content": turn["question"]})
        messages.append({"role": "assistant", "content": turn["answer"]})
    messages.append({
        "role": "user",
        "content": f"Contexto:\n{context}\n\nPregunta: {question}"
    })
    return _call_llm(messages)


def check_hallucination(context: str, question: str, answer: str) -> dict:
    """Ejecuta LettuceDetect y devuelve los spans alucinados + veredicto booleano."""
    spans = halluc_detector.predict(
        context=[context],      # espera lista de strings
        question=question,
        answer=answer,
        output_format="spans",
    )
    return {"hallucinated": len(spans) > 0, "spans": spans}


def _print_hallucination(result: dict) -> None:
    """Imprime el resultado de LettuceDetect de forma legible."""
    print("\nLettuceDetect:")
    if not result["hallucinated"]:
        print("  Sin alucinaciones detectadas.")
    else:
        print(f"  {len(result['spans'])} fragmento(s) sospechoso(s):")
        for span in result["spans"]:
            conf = span.get("confidence", 0)
            text = span.get("text", "").strip()
            print(f"    [{conf:.1%}] '{text}'")


print("Funciones del pipeline listas")



In [ ]:
# %% [7. Funcion rag_query — pregunta puntual sin historial]

def rag_query(question: str, show_context: bool = False) -> None:
    """Pipeline completo para una pregunta puntual (sin historial)."""
    print(f"\n{'='*60}")
    print(f"Pregunta: {question}")
    print("=" * 60)

    chunks_libro, chunks_resumenes = retrieve(question)
    context = build_context(chunks_libro, chunks_resumenes)

    if show_context:
        print("\nContexto recuperado:")
        print(context[:2000], "..." if len(context) > 2000 else "")

    answer = generate_answer(question, context)
    print(f"\nRespuesta:\n{answer}")

    result = check_hallucination(context, question, answer)
    _print_hallucination(result)
    print("=" * 60)


In [ ]:
# %% [8. Modo conversacion con historial]

chat_history: list = []


def chat(question: str, show_context: bool = False) -> None:
    """Un turno de conversacion con historial + deteccion de alucinaciones."""
    global chat_history

    print(f"\n{'='*60}")
    print(f"Tu: {question}")

    chunks_libro, chunks_resumenes = retrieve(question)
    context = build_context(chunks_libro, chunks_resumenes)

    if show_context:
        print("\nContexto:")
        print(context[:1500], "..." if len(context) > 1500 else "")

    answer = generate_answer_with_history(question, context, chat_history)
    print(f"\nBot: {answer}")

    result = check_hallucination(context, question, answer)
    _print_hallucination(result)

    chat_history.append({"question": question, "answer": answer})
    print("=" * 60)


def reset_chat() -> None:
    """Reinicia el historial de la conversacion."""
    global chat_history
    chat_history = []
    print("Historial reiniciado.")

In [ ]:
# %% [10. Loop interactivo de conversacion]

if __name__ == "__main__":
    reset_chat()
    print("\nChatbot GoT activo.")
    print("  'reset'  -> reinicia el historial")
    print("  'salir'  -> termina la sesion\n")

    while True:
        try:
            user_input = input("Tu: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSesion finalizada.")
            break

        if not user_input:
            continue
        if user_input.lower() == "salir":
            print("Hasta la proxima!")
            break
        if user_input.lower() == "reset":
            reset_chat()
            continue

        chat(user_input)